In [ ]:
import pandas as pd
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib import pubmed, gpt

In [ ]:
importlib.reload(gpt)

## Read and compare

In [ ]:
DIR = r"C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\\"

In [ ]:
selected_articles = pd.read_excel(DIR + 'final30.xlsx')
selected_articles.head()

In [ ]:
prescreened_articles = pd.read_excel(DIR + 'prescreaned1501.xlsx')
prescreened_articles.head()

In [ ]:
selected_articles.rename(columns={'Article Title': 'Title'}, inplace=True)

In [ ]:
len(selected_articles), selected_articles['Title'].nunique()

In [ ]:
len(prescreened_articles), prescreened_articles['Title'].nunique()

In [ ]:
selected_articles['Title'].isin(prescreened_articles['Title']).sum()

In [ ]:
selected_titles = {extract_letters(t) for t in selected_articles['Title']}
print(len(selected_titles))
all_titles = {extract_letters(t) for t in prescreened_articles['Title']}
print(len(all_titles))

In [ ]:
len(selected_titles & all_titles)

In [ ]:
selected_articles['title_extract'] = selected_articles['Title'].map(extract_letters)
missing_titles = selected_articles.loc[~selected_articles['title_extract'].isin(all_titles), 'Title']
missing_titles

In [ ]:
selected_articles['found'] = selected_articles['title_extract'].isin(all_titles)

## Add abstracts from pubmed

In [ ]:
paul = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\paul100titles.xlsx')

In [ ]:
paul

In [ ]:
res = []
    
for title in tqdm(paul.title):
    query_str = f'"{title}"'
    res.append(pubmed.query(query_str, max_results=5))

In [ ]:
res = pubmed.query("What’s going on following acute covid-19? Clinical characteristics of patients in an outpatient rehabilitation program", max_results=5)
len(res)

In [ ]:
res

In [ ]:
from pathlib import Path

# 1. Combine all of the PubMed‐query DataFrames into one big DataFrame
#    (skipping any entries that aren’t DataFrames)
df_res = pd.concat(
    [df for df in res if isinstance(df, pd.DataFrame)],
    ignore_index=True
)

# 2. Define and create the output directory
out_dir  = Path(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables')
out_dir.mkdir(parents=True, exist_ok=True)

# 3. Save to Excel
output_path = out_dir / 'not_selected_titles_with_pubmed_results.xlsx'
df_res.to_excel(output_path, index=False)

print(f"✅ Saved {len(df_res)} PubMed records to:\n   {output_path}")


In [ ]:
# --- Load your existing DataFrame `df_res` (if not already in the notebook) ---
# For example:
# df_res = pd.read_pickle('df_res.pkl')

# Load the SR table from Excel
df_sr = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\SR_files_with_pubmed_and_title.xlsx')

# Define the common key column (adjust if your column names differ)
key = 'PubMedID'

# Add indicator columns
df_res['in_SR'] = df_res[key].isin(df_sr[key])
df_sr['in_df_res'] = df_sr[key].isin(df_res[key])

# Save the updated tables back to Excel
df_res.to_excel('df_res_with_indicator.xlsx', index=False)
df_sr.to_excel('SR_files_with_indicator.xlsx', index=False)

# Display a sample to verify
df_res.head(), df_sr.head()


In [ ]:
res.PMID.dtype

In [ ]:
ids = prescreened_articles.PMID[:10]
print(len(ids))
res = []

start = time.time()
for pmid in tqdm(ids):
    res.append(pubmed.query(str(pmid), max_results=5))
print(f'{time.time() - start:.21f}sec')

In [ ]:
BATCH_SIZE = 100
ids = prescreened_articles.PMID[:BATCH_SIZE]

start = time.time()
query_str = ' OR '.join(map(str, ids))
res = pubmed.query(query_str, max_results=BATCH_SIZE * 2)
print(f'{time.time() - start:.21f}sec')
print(len(res))
print(set(res.pubmed_id) == set(ids))

In [ ]:
set(res.pubmed_id) - set(map(str, ids))

In [ ]:
set(map(str, ids)) - set(res.pubmed_id)

In [ ]:
BATCH_SIZE = 300
ids = prescreened_articles.PMID[:BATCH_SIZE]
ids = [str(i) for i in ids]

start = time.time()
query_str = ' OR '.join(ids)
res = pubmed.query(query_str, max_results=BATCH_SIZE * 2)
print(f'{time.time() - start:.21f}sec')
print(len(res))
print(set(res.pubmed_id) == set(ids))

In [ ]:
set(res.pubmed_id) - set(ids)

In [ ]:
parts = []
starts = list(range(0, len(prescreened_articles), BATCH_SIZE))
for start in tqdm(starts):
    ids = prescreened_articles.PMID[start:start + BATCH_SIZE]
    query_str = ' OR '.join(map(str, ids))
    part = pubmed.query(query_str, max_results=BATCH_SIZE * 2)
    parts.append(part)
all_pubmed_results = pd.concat(parts, axis=0)

In [ ]:
len(all_pubmed_results)

In [ ]:
all_pubmed_results['pubmed_id'] = all_pubmed_results['pubmed_id'].astype(int)

In [ ]:
sum(prescreened_articles.PMID.isin(all_pubmed_results['pubmed_id']))

In [ ]:
big_with_pubmed = pd.merge(prescreened_articles, all_pubmed_results, left_on='PMID', right_on='pubmed_id', how='left')
big_with_pubmed.head()

In [ ]:
big_with_pubmed[big_with_pubmed.pubmed_id.isna()]

In [ ]:
# Don't run this, bad idea!

for ind, row in big_with_pubmed[big_with_pubmed.pubmed_id.isna()].iterrows():
    res = pubmed.query(row.PMID, max_results=1)
    res['pubmed_id'] = res['pubmed_id'].astype(int)
    big_with_pubmed.loc[ind, res.columns] = res.loc[0, :]
big_with_pubmed.head()

In [ ]:
big_with_pubmed['title'] = big_with_pubmed['title'].fillna('')
big_with_pubmed['title_extract'] = big_with_pubmed['Title'].map(extract_letters)
big_with_pubmed['title_extract2'] = big_with_pubmed['title'].map(extract_letters)

In [ ]:
(big_with_pubmed.title_extract != big_with_pubmed.title_extract2).sum()

In [ ]:
big_with_pubmed.loc[big_with_pubmed.title_extract != big_with_pubmed.title_extract2, ['Title', 'title']]

In [ ]:
big_with_pubmed.loc[958]

In [ ]:
big_with_pubmed.loc[1128]

In [ ]:
selected_articles['found1'] = selected_articles['title_extract'].isin(big_with_pubmed['title_extract'])
selected_articles['found2'] = selected_articles['title_extract'].isin(big_with_pubmed['title_extract2'])
selected_articles['found'] = selected_articles['found1'] | selected_articles['found2']
selected_articles['found'].sum()

In [ ]:
del selected_articles['found1']
del selected_articles['found2']

In [ ]:
df.title_extract.dtype, selected_articles.title_extract.dtype

In [ ]:
df = big_with_pubmed[['title_extract', 'PMID', 'abstract']].rename(columns={'PMID': 'pubmed_id'})
selected_articles_with_abstracts = pd.merge(selected_articles, df, on='title_extract', how='left')
selected_articles_with_abstracts

In [ ]:
del selected_articles_with_abstracts['title_extract']

In [ ]:
big_with_pubmed.columns

In [ ]:
del big_with_pubmed['pubmed_id']
del big_with_pubmed['title_extract']
del big_with_pubmed['title_extract2']

In [ ]:
big_with_pubmed.to_excel(DIR + 'prescreened_with_pubmed.xlsx', index=False)

In [ ]:
missing_pmid_indices = list(selected_articles_with_abstracts[selected_articles_with_abstracts.pubmed_id.isna()].index)
len(missing_pmid_indices)

In [ ]:
found_results = []
for ind in tqdm(missing_pmid_indices):
    orig_title = selected_articles_with_abstracts.loc[ind, 'Title']
    title = orig_title.replace('"', '')
    res = pubmed.query(f'"{title}"', max_results=3)
    found_results.append(res)
    print(len(res), orig_title)

In [ ]:
for ind, res in zip(missing_pmid_indices, found_results):
    if len(res) == 1:
        res['pubmed_id'] = res['pubmed_id'].astype(int)
        selected_articles_with_abstracts.loc[ind, ['pubmed_id', 'abstract']] = res.loc[0, ['pubmed_id', 'abstract']]

In [ ]:
selected_articles_with_abstracts.to_excel(DIR + 'final30_with_abstracts.xlsx', index=False)

## Run ChatGPT

In [ ]:
selected_articles_with_abstracts = pd.read_excel(DIR + 'final30_with_manually_filled_abstracts.xlsx')

In [ ]:
SYSTEM_PROMPT = '''
You are a scientist. You are given a set of inclusion and exclusion criteria and a list of articles. You must decide
whether to include each article and explain why. 
Try your best even if the abstract is missing.

The format should be as follows:

```
## ARTICLE 1

<reasoning>

INCLUDE or EXCLUDE
```

You must make a decision for **each** input and write it only once.
'''

In [ ]:
renamings = {}
for col in selected_articles_with_abstracts.columns:
    renamings[col] = col.strip()
selected_articles_with_abstracts.rename(columns=renamings, inplace=True)

In [ ]:
selected_eligibility = gpt.check_eligibility(
    selected_articles_with_abstracts,
    model=gpt.Models.MINI,
    system_prompt=SYSTEM_PROMPT,
    criteria=CRITERIA,
)
print(selected_eligibility.decision.sum())
selected_eligibility.head()

In [ ]:
selected_eligibility.to_excel(DIR + 'final30_with_eligibility.xlsx', index=False)

In [ ]:
big_with_pubmed.rename(columns={'Publication Year': 'Year'}, inplace=True)

In [ ]:
big_eligibility = gpt.check_eligibility(
    big_with_pubmed,
    model=gpt.Models.MINI,
    system_prompt=SYSTEM_PROMPT,
    criteria=CRITERIA,
)
print(big_eligibility.decision.sum(), big_eligibility.decision.mean())
big_eligibility.head()

In [ ]:
big_eligibility.to_excel(DIR + 'prescreened_with_eligibility.xlsx', index=False)

## Fix bad abstracts

In [ ]:
selected_df = pd.read_excel(DIR + 'final30_with_eligibility.xlsx')

In [ ]:
df = selected_df[selected_df['need to redo']]
df_with_eligibility = gpt.check_eligibility(
    df,
    model=gpt.Models.MINI,
    system_prompt=SYSTEM_PROMPT,
    criteria=CRITERIA,
)
print(len(df_with_eligibility))
df_with_eligibility.head()

In [ ]:
selected_df.loc[selected_df['need to redo'], ['reasoning', 'decision']] = df_with_eligibility[['reasoning', 'decision']]
selected_df.decision.sum()

In [ ]:
selected_df.to_excel(DIR + 'final30_with_eligibility_2.xlsx', index=False)

## Not selected

In [ ]:
not_selected_df = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\paul74.xlsx')
len(not_selected_df)

In [ ]:
SYSTEM_PROMPT = '''
You are a scientist. You are given a set of inclusion and exclusion criteria and a list of articles. You must decide
whether to include each article and explain why. 
Try your best even if the abstract is missing.

The format should be as follows:

```
## ARTICLE 1

<reasoning>

INCLUDE or EXCLUDE
```

You must make a decision for **each** input and write it only once.
'''

In [ ]:
CRITERIA = '''
## Inclusion

*	Peer-reviewed observational studies (cohort, cross-sectional, case-control)
*	Laboratory-confirmed SARS-CoV-2 cases (serology or molecular)
*	Report post-acute (persistent or new) physical or mental-health symptoms after recovery
*	Any follow-up duration; no time-of-onset restriction
*	English-language; any geography

## Exclusion

*	Case reports/series, RCTs, reviews, editorials, letters, protocols
*	Studies of acute-phase symptoms only
*	Non-peer-reviewed or non-English publications
'''

In [ ]:
not_selected_eligibility = gpt.check_eligibility(
    not_selected_df,
    model=gpt.Models.MINI,
    system_prompt=SYSTEM_PROMPT,
    criteria=CRITERIA,
)
not_selected_eligibility.head()

In [ ]:
not_selected_eligibility.decision.sum()

In [ ]:
big_df = pd.read_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\SR_files_with_pubmed_and_title.xlsx')
big_df.columns

In [ ]:
not_selected_eligibility['title_extract'] = not_selected_eligibility.title.apply(extract_letters)
print(not_selected_eligibility['title_extract'].nunique(), len(not_selected_eligibility))
big_df['title_extract'] = big_df.title.apply(extract_letters)
not_selected_eligibility['found'] = not_selected_eligibility.title_extract.isin(big_df.title_extract)
not_selected_eligibility.found.sum()

In [ ]:
not_selected_eligibility.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\not_selected_eligibility.xlsx')

In [ ]:
len(big_df)